# Self-Derived Skills: Colab runner

Before running this notebook:

1. Select a GPU in **Runtime > Change runtime type**.
2. Create a fine-grained GitHub token limited to this repository, with read-only Contents access.
3. Add it in Colab's **Secrets** panel (the key icon) as `GH_TOKEN`, and allow this notebook to use it.

Then run the cells in order. The probe writes to the runtime's local disk, so its files disappear when the session ends.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
import subprocess

from google.colab import userdata

REPO_URL = "https://github.com/Ruichu-Xia/self-derived-skills.git"
REPO_DIR = "/content/self-derived-skills"
RUN_ROOT = "/content/runs"

# The token stays in an environment variable so that it never appears in a command, URL, or error.
GIT_ENV = {**os.environ, "GH_TOKEN": userdata.get("GH_TOKEN")}
CREDENTIAL_HELPER = (
    "credential.helper=!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"
)


def git(*args, cwd=None):
    """Run a git command that authenticates with the token in the environment."""
    subprocess.run(["git", "-c", CREDENTIAL_HELPER, *args], env=GIT_ENV, cwd=cwd, check=True)


if os.path.isdir(REPO_DIR):
    git("pull", "--ff-only", cwd=REPO_DIR)
else:
    git("clone", REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
!git log -1 --format='%h %s'

In [ ]:
%pip install -q -e ".[train,cuda]"
# Colab preinstalls a torchao too old for PEFT, which then crashes. This project does not use it.
%pip uninstall -q -y torchao
!python -c "import torch, transformers, trl; print('torch', torch.__version__, 'transformers', transformers.__version__, 'trl', trl.__version__, 'cuda', torch.cuda.is_available())"

## GRPO probe

Checks that the model loads, that left padding does not change its outputs, and how long a real GRPO step takes. It runs 6 training steps and trains nothing useful. Send the printed summary back when it finishes.

In [ ]:
from datetime import datetime

PROBE_DIR = f"{RUN_ROOT}/probe-{datetime.now():%Y%m%d-%H%M%S}"
!python experiments/probe_grpo.py --config configs/grpo_math.yaml --output-dir "{PROBE_DIR}" --max-steps 6
print(open(f"{PROBE_DIR}/probe_summary.json").read())